# 01 什么是神经网络

这一节先不急着写复杂模型，而是把神经网络的核心直觉建立起来：神经网络本质上是一个可以通过数据自动调整参数的函数。

本节参考黑马程序员《神经网络与深度学习》课程的学习主线：先理解深度学习的发展与应用，再进入 PyTorch 框架、人工神经网络、反向传播、CNN 图像任务和 RNN 序列任务。


## 1. 学习目标

学完这一节，需要能回答下面几个问题：

1. 神经网络为什么可以看成一个函数？
2. 一个神经元到底做了什么计算？
3. 权重、偏置、激活函数分别有什么作用？
4. 训练神经网络时，损失函数、反向传播、优化器分别负责什么？
5. 为什么后面要继续学习 MLP、BP、CNN、RNN？


## 2. 神经网络是一类可训练函数

普通函数通常由人手写规则，例如：

$$
y = 2x + 1
$$

神经网络也是函数，只不过函数里的参数不是手工指定，而是从数据中学习出来。可以把一个神经网络记作：

$$
\hat{y} = f_{\theta}(x)
$$

其中：

- $x$ 表示输入数据，比如图片、文本、表格特征。
- $\hat{y}$ 表示模型预测结果。
- $f_{\theta}$ 表示带参数的函数。
- $\theta$ 表示模型中所有可学习参数，主要包括权重和偏置。

训练神经网络，就是不断调整 $\theta$，让 $\hat{y}$ 越来越接近真实标签 $y$。


## 3. 一个神经元在计算什么

神经元可以先理解成三个动作：加权、求和、激活。

如果输入是一个向量：

$$
\mathbf{x} = [x_1, x_2, \dots, x_n]
$$

权重是：

$$
\mathbf{w} = [w_1, w_2, \dots, w_n]
$$

偏置是：

$$
b
$$

那么神经元先计算线性部分：

$$
z = \mathbf{w}^{T}\mathbf{x} + b = \sum_{i=1}^{n} w_i x_i + b
$$

然后通过激活函数得到输出：

$$
a = \sigma(z)
$$

其中 $\sigma$ 表示激活函数，$a$ 表示神经元的输出。


## 4. 权重、偏置、激活函数的直觉

- 权重 $w_i$：决定第 $i$ 个输入特征有多重要。
- 偏置 $b$：让模型的判断边界可以平移，不必强行经过原点。
- 激活函数 $\sigma$：给模型加入非线性能力。

如果没有激活函数，多层线性变换叠在一起，本质上仍然只是一个线性变换。比如：

$$
\mathbf{y} = \mathbf{W}_2(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2
$$

展开后仍然可以写成：

$$
\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}
$$

所以，激活函数是神经网络能够拟合复杂关系的关键之一。


## 5. 常见激活函数

Sigmoid 函数常用于早期神经网络和二分类输出层：

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Tanh 函数会把输出压到 $(-1, 1)$：

$$
\tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}
$$

ReLU 是深度学习中非常常见的激活函数：

$$
\operatorname{ReLU}(z) = \max(0, z)
$$

Softmax 常用于多分类输出层，把多个分数转换成概率分布：

$$
\operatorname{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

其中 $K$ 表示类别数量，$z_i$ 表示第 $i$ 个类别的原始分数。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 400)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid, label='Sigmoid')
plt.plot(z, tanh, label='Tanh')
plt.plot(z, relu, label='ReLU')
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Activation Functions')
plt.xlabel('z')
plt.ylabel('activation(z)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 6. 从单个神经元到多层网络

多个神经元排在一起，就是一层网络。多层网络可以写成：

$$
\mathbf{h}^{(1)} = \sigma(\mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)})
$$

$$
\mathbf{h}^{(2)} = \sigma(\mathbf{W}^{(2)}\mathbf{h}^{(1)} + \mathbf{b}^{(2)})
$$

$$
\hat{\mathbf{y}} = g(\mathbf{W}^{(3)}\mathbf{h}^{(2)} + \mathbf{b}^{(3)})
$$

其中：

- $\mathbf{x}$ 是输入层。
- $\mathbf{h}^{(1)}$ 和 $\mathbf{h}^{(2)}$ 是隐藏层。
- $\hat{\mathbf{y}}$ 是输出层。
- $g$ 是输出层激活函数，具体选择取决于任务。

这种结构通常叫多层感知机，也就是 MLP。后面学习 BP 神经网络时，会重点研究它如何训练。


## 7. 训练神经网络的四个核心角色

神经网络训练可以先拆成四个角色：

1. 前向传播：用当前参数算出预测值 $\hat{y}$。
2. 损失函数：衡量预测值 $\hat{y}$ 和真实值 $y$ 的差距。
3. 反向传播：计算损失对每个参数的梯度。
4. 优化器：根据梯度更新参数。

以回归任务常见的均方误差为例：

$$
\mathcal{L}(\hat{y}, y) = \frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})^2
$$

参数更新可以先理解成梯度下降：

$$
\theta \leftarrow \theta - \eta \frac{\partial \mathcal{L}}{\partial \theta}
$$

其中 $\eta$ 是学习率，$\frac{\partial \mathcal{L}}{\partial \theta}$ 是损失函数对参数 $\theta$ 的梯度。


## 8. 小实验：手算一个神经元

假设一个样本有两个特征：

$$
\mathbf{x} = [0.8, 0.4]
$$

神经元参数为：

$$
\mathbf{w} = [0.5, -0.3], \quad b = 0.1
$$

先计算：

$$
z = 0.5 \times 0.8 + (-0.3) \times 0.4 + 0.1
$$

再经过 Sigmoid：

$$
a = \frac{1}{1 + e^{-z}}
$$

下面用 NumPy 验证一下。


In [ ]:
x = np.array([0.8, 0.4])
w = np.array([0.5, -0.3])
b = 0.1

z = np.dot(w, x) + b
a = 1 / (1 + np.exp(-z))

print('z =', z)
print('a =', a)


## 9. 为什么需要多层网络

线性模型擅长处理线性可分问题，但很多真实任务并不是简单直线、平面或超平面能分开的。

例如异或问题的规则是：两个输入不同则输出 $1$，两个输入相同则输出 $0$。

$$
\begin{array}{c|c|c}
x_1 & x_2 & y \\
\hline
0 & 0 & 0 \\
0 & 1 & 1 \\
1 & 0 & 1 \\
1 & 1 & 0
\end{array}
$$

这个问题无法被一个简单线性分类器完美分开，但可以用带隐藏层的神经网络表达。


In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([0, 1, 1, 0])

def step(v):
    return (v > 0).astype(int)

# 隐藏层第一个神经元学习 OR，第二个神经元学习 NAND。
W1 = np.array([
    [1, 1],
    [-1, -1]
])
b1 = np.array([-0.5, 1.5])

# 输出层学习 AND。
W2 = np.array([1, 1])
b2 = -1.5

H = step(X @ W1.T + b1)
y_hat = step(H @ W2 + b2)

print('输入 X:')
print(X)
print('\n隐藏层 H:')
print(H)
print('\n预测 y_hat:', y_hat)
print('真实 y:   ', y)


## 10. 神经网络学习路径

结合当前目录里已经完成的 PyTorch 基础内容，后面可以按下面顺序继续：

1. 神经元、感知机与 MLP。
2. 损失函数、梯度下降与反向传播。
3. 使用 PyTorch 实现二分类 MLP。
4. 使用 MNIST 或 Fashion-MNIST 完成多分类任务。
5. 学习 CNN，进入图像分类任务。
6. 学习 RNN、LSTM、GRU，进入序列建模与文本生成任务。
7. 学习 Attention 与 Transformer，为 NLP 和大模型打基础。


## 11. 本节总结

这一节先记住三句话：

1. 神经网络是带参数的函数，核心形式是 $\hat{y} = f_{\theta}(x)$。
2. 神经元的基本计算是 $a = \sigma(\mathbf{w}^{T}\mathbf{x} + b)$。
3. 训练的目标是最小化损失函数 $\mathcal{L}$，通过梯度下降更新参数 $\theta$。

下一节建议进入：感知机与多层感知机，重点理解为什么单层模型不够、隐藏层如何提升表达能力。
